# 🧠 Soft Computing Project
## Fuzzy Logic Sentiment Analysis on Amazon Product Reviews Dataset
### Dataset: `1429_1.csv` — 34,660 Amazon Product Reviews

---

| Component | Detail |
|---|---|
| **Technique** | Soft Computing — Fuzzy Logic |
| **FIS Type** | Mamdani Fuzzy Inference System |
| **MF Type** | Triangular (trimf) |
| **Rules** | 15 IF-THEN linguistic rules |
| **Defuzzification** | Centroid (Center of Gravity) |
| **Dataset** | Amazon Product Reviews (34,660 rows) |
| **Task** | Sentiment classification from review text |

## ✅ Step 1: Install & Import Libraries

In [ ]:
import subprocess, sys
pkgs = ['scikit-fuzzy','numpy','matplotlib','pandas','scikit-learn','seaborn']
for p in pkgs:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('✅ All packages ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import re, warnings, os
warnings.filterwarnings('ignore')

# Dark theme plots
plt.rcParams.update({
    'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0',
    'xtick.color':'#94a3b8','ytick.color':'#94a3b8',
    'text.color':'#e2e8f0','grid.color':'#334155','grid.alpha':0.5,'font.size':11
})
os.makedirs('../outputs', exist_ok=True)
print(f'✅ numpy {np.__version__}  |  skfuzzy {fuzz.__version__}  |  pandas {pd.__version__}')

## ✅ Step 2: Load & Explore the Dataset

In [ ]:
# Load the Amazon reviews dataset
CSV_PATH = '../data/reviews.csv'   # adjust if needed
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'📦 Raw dataset shape: {df_raw.shape}')
print(f'📋 Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# ── Keep only essential columns ──────────────────────────────
df = df_raw[['name','reviews.text','reviews.title','reviews.rating',
             'reviews.doRecommend','reviews.numHelpful','brand']].copy()
df.columns = ['product','review_text','review_title','rating',
              'recommend','num_helpful','brand']

# Drop rows with no review text or rating
df.dropna(subset=['review_text','rating'], inplace=True)
df['rating']      = pd.to_numeric(df['rating'], errors='coerce')
df.dropna(subset=['rating'], inplace=True)
df['review_text'] = df['review_text'].astype(str).str.strip()
df = df[df['review_text'].str.len() > 10].reset_index(drop=True)

print(f'✅ Clean dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'   Rating distribution:')
print(df['rating'].value_counts().sort_index().to_string())
print(f'\n   Unique products: {df["product"].nunique()}')

In [ ]:
# Dataset EDA visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dataset Exploration — Amazon Product Reviews', 
             fontsize=14, fontweight='bold', color='#38bdf8')

# Rating distribution
ax = axes[0]
colors = ['#ef4444','#f97316','#facc15','#86efac','#22c55e']
counts = df['rating'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='#0f172a', width=0.7)
ax.set_title('Rating Distribution', color='#38bdf8', fontweight='bold')
ax.set_xlabel('Star Rating'); ax.set_ylabel('Count')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+200, 
            f'{int(b.get_height()):,}', ha='center', fontsize=9, color='#94a3b8')
ax.grid(True, axis='y', alpha=0.3)

# Review text length distribution
ax = axes[1]
df['text_len'] = df['review_text'].str.len()
ax.hist(df['text_len'].clip(0, 1000), bins=40, color='#38bdf8', edgecolor='#0f172a', alpha=0.8)
ax.set_title('Review Length Distribution', color='#38bdf8', fontweight='bold')
ax.set_xlabel('Characters'); ax.set_ylabel('Count')
ax.axvline(df['text_len'].median(), color='#facc15', linestyle='--', linewidth=1.5, label=f'Median: {df["text_len"].median():.0f}')
ax.legend(); ax.grid(True, alpha=0.3)

# Recommend distribution
ax = axes[2]
rec = df['recommend'].fillna('Unknown').value_counts()
ax.pie(rec.values, labels=rec.index, autopct='%1.1f%%',
       colors=['#22c55e','#ef4444','#64748b'],
       textprops={'color':'#e2e8f0'},
       wedgeprops={'edgecolor':'#0f172a','linewidth':2})
ax.set_title('Recommend Distribution', color='#38bdf8', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/eda_overview.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ EDA plots saved!')

## ✅ Step 3: Define Fuzzy Universe of Discourse

In [ ]:
# Universe of Discourse
universe           = np.arange(0, 10.01, 0.1)
sentiment_universe = np.arange(0, 100.01, 0.5)

print('📐 Universe of Discourse')
print(f'   Input  : [0, 10]   — normalized positive/negative/punctuation scores')
print(f'   Output : [0, 100]  — final fuzzy sentiment score')
print(f'   Input points  : {len(universe)}')
print(f'   Output points : {len(sentiment_universe)}')

## ✅ Step 4: Define Fuzzy Linguistic Variables & Membership Functions

In [ ]:
# ── Antecedents (Inputs) ─────────────────────────────────────
pos_score   = ctrl.Antecedent(universe, 'positive_score')
neg_score   = ctrl.Antecedent(universe, 'negative_score')
punct_score = ctrl.Antecedent(universe, 'punctuation_score')

# ── Consequent (Output) ──────────────────────────────────────
sentiment = ctrl.Consequent(sentiment_universe, 'sentiment')

# ── Triangular Membership Functions ─────────────────────────
# Positive Score: low / medium / high
pos_score['low']    = fuzz.trimf(universe, [0,  0,  4])
pos_score['medium'] = fuzz.trimf(universe, [2,  5,  8])
pos_score['high']   = fuzz.trimf(universe, [6, 10, 10])

# Negative Score: low / medium / high
neg_score['low']    = fuzz.trimf(universe, [0,  0,  4])
neg_score['medium'] = fuzz.trimf(universe, [2,  5,  8])
neg_score['high']   = fuzz.trimf(universe, [6, 10, 10])

# Punctuation Score: low / high
punct_score['low']  = fuzz.trimf(universe, [0,  0,  5])
punct_score['high'] = fuzz.trimf(universe, [4, 10, 10])

# Output Sentiment: 5 linguistic terms
sentiment['very_negative'] = fuzz.trimf(sentiment_universe, [0,   0,  20])
sentiment['negative']      = fuzz.trimf(sentiment_universe, [10, 25,  40])
sentiment['neutral']       = fuzz.trimf(sentiment_universe, [30, 50,  70])
sentiment['positive']      = fuzz.trimf(sentiment_universe, [60, 75,  90])
sentiment['very_positive'] = fuzz.trimf(sentiment_universe, [80,100, 100])

print('✅ Fuzzy variables & membership functions defined!')
print('   Input  vars: positive_score, negative_score, punctuation_score')
print('   Output var : sentiment → [very_negative, negative, neutral, positive, very_positive]')

## ✅ Step 5: Visualize Membership Functions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Fuzzy Membership Functions — Triangular (trimf)', 
             fontsize=15, fontweight='bold', color='#38bdf8')

C = {'low':'#f87171','medium':'#facc15','high':'#4ade80',
     'very_negative':'#ef4444','negative':'#f97316',
     'neutral':'#facc15','positive':'#86efac','very_positive':'#22c55e'}

def style(ax, title, xlabel):
    ax.set_facecolor('#1e293b'); ax.set_title(title,color='#38bdf8',fontweight='bold')
    ax.set_xlabel(xlabel,color='#94a3b8'); ax.set_ylabel('μ(x)',color='#94a3b8')
    ax.set_ylim(-0.05,1.15); ax.grid(True,color='#334155',alpha=0.4)
    ax.tick_params(colors='#94a3b8')
    for s in ax.spines.values(): s.set_edgecolor('#334155')

for t in ['low','medium','high']:
    axes[0,0].plot(universe, pos_score[t].mf, color=C[t], lw=2.5, label=t)
    axes[0,0].fill_between(universe, pos_score[t].mf, alpha=0.12, color=C[t])
style(axes[0,0],'Input: Positive Score','Score (0–10)')
axes[0,0].legend()

for t in ['low','medium','high']:
    axes[0,1].plot(universe, neg_score[t].mf, color=C[t], lw=2.5, label=t)
    axes[0,1].fill_between(universe, neg_score[t].mf, alpha=0.12, color=C[t])
style(axes[0,1],'Input: Negative Score','Score (0–10)')
axes[0,1].legend()

axes[1,0].plot(universe, punct_score['low'].mf,  color='#f87171', lw=2.5, label='low')
axes[1,0].fill_between(universe, punct_score['low'].mf,  alpha=0.12, color='#f87171')
axes[1,0].plot(universe, punct_score['high'].mf, color='#4ade80', lw=2.5, label='high')
axes[1,0].fill_between(universe, punct_score['high'].mf, alpha=0.12, color='#4ade80')
style(axes[1,0],'Input: Punctuation Score','Score (0–10)')
axes[1,0].legend()

for t in ['very_negative','negative','neutral','positive','very_positive']:
    axes[1,1].plot(sentiment_universe, sentiment[t].mf, color=C[t], lw=2.5, label=t.replace('_',' ').title())
    axes[1,1].fill_between(sentiment_universe, sentiment[t].mf, alpha=0.1, color=C[t])
style(axes[1,1],'Output: Sentiment Score','Sentiment Score (0–100)')
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/membership_functions.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('✅ Saved: outputs/membership_functions.png')

## ✅ Step 6: Define Fuzzy Rule Base & Build FIS

In [ ]:
rules = [
    # Strong positive
    ctrl.Rule(pos_score['high']   & neg_score['low'],                           sentiment['very_positive']),
    ctrl.Rule(pos_score['high']   & neg_score['low']   & punct_score['high'],   sentiment['very_positive']),
    ctrl.Rule(pos_score['high']   & neg_score['medium'],                        sentiment['positive']),
    ctrl.Rule(pos_score['medium'] & neg_score['low'],                           sentiment['positive']),
    ctrl.Rule(pos_score['medium'] & neg_score['low']   & punct_score['high'],   sentiment['positive']),
    # Neutral
    ctrl.Rule(pos_score['medium'] & neg_score['medium'],                        sentiment['neutral']),
    ctrl.Rule(pos_score['low']    & neg_score['low'],                           sentiment['neutral']),
    ctrl.Rule(pos_score['low']    & neg_score['low']   & punct_score['low'],    sentiment['neutral']),
    # Strong negative
    ctrl.Rule(neg_score['high']   & pos_score['low'],                           sentiment['very_negative']),
    ctrl.Rule(neg_score['high']   & pos_score['medium'],                        sentiment['negative']),
    ctrl.Rule(neg_score['medium'] & pos_score['low'],                           sentiment['negative']),
    ctrl.Rule(neg_score['high']   & punct_score['high'],                        sentiment['very_negative']),
    # Mixed
    ctrl.Rule(pos_score['high']   & neg_score['high'],                          sentiment['neutral']),
    ctrl.Rule(pos_score['low']    & neg_score['medium'],                        sentiment['negative']),
    ctrl.Rule(pos_score['medium'] & neg_score['high'],                          sentiment['negative']),
]

fis_ctrl = ctrl.ControlSystem(rules)
fis_sim  = ctrl.ControlSystemSimulation(fis_ctrl)

print('🧠 Mamdani FIS built!')
rule_table = [
    ('R1',  'pos=HIGH   & neg=LOW',                  'VERY POSITIVE'),
    ('R2',  'pos=HIGH   & neg=LOW  & punct=HIGH',    'VERY POSITIVE'),
    ('R3',  'pos=HIGH   & neg=MED',                  'POSITIVE'),
    ('R4',  'pos=MED    & neg=LOW',                  'POSITIVE'),
    ('R5',  'pos=MED    & neg=LOW  & punct=HIGH',    'POSITIVE'),
    ('R6',  'pos=MED    & neg=MED',                  'NEUTRAL'),
    ('R7',  'pos=LOW    & neg=LOW',                  'NEUTRAL'),
    ('R8',  'pos=LOW    & neg=LOW  & punct=LOW',     'NEUTRAL'),
    ('R9',  'neg=HIGH   & pos=LOW',                  'VERY NEGATIVE'),
    ('R10', 'neg=HIGH   & pos=MED',                  'NEGATIVE'),
    ('R11', 'neg=MED    & pos=LOW',                  'NEGATIVE'),
    ('R12', 'neg=HIGH   & punct=HIGH',               'VERY NEGATIVE'),
    ('R13', 'pos=HIGH   & neg=HIGH',                 'NEUTRAL'),
    ('R14', 'pos=LOW    & neg=MED',                  'NEGATIVE'),
    ('R15', 'pos=MED    & neg=HIGH',                 'NEGATIVE'),
]
print('\n📋 Fuzzy Rule Base:')
print(f'{"ID":<5} {"IF":<40} {"THEN":<15}')
print('─'*60)
for r in rule_table:
    print(f'{r[0]:<5} {r[1]:<40} → {r[2]}')

## ✅ Step 7: Feature Extraction from Review Text

In [ ]:
POSITIVE_WORDS = {
    'good','great','excellent','amazing','wonderful','fantastic','love','happy',
    'best','awesome','beautiful','nice','perfect','brilliant','outstanding',
    'superb','terrific','pleasant','enjoy','enjoyed','positive','helpful',
    'recommend','glad','delighted','impressive','marvelous','fabulous','splendid',
    'magnificent','joyful','cheerful','pleased','satisfied','thrilled','excited',
    'fun','incredible','genius','smooth','clean','fresh','powerful','elegant',
    'efficient','easy','fast','friendly','kind','light','quality','crisp',
    'clear','bright','solid','durable','affordable','value','worth','quick'
}
NEGATIVE_WORDS = {
    'bad','terrible','awful','horrible','worst','hate','ugly','disgusting',
    'disappointing','poor','boring','annoying','frustrating','useless','waste',
    'broken','slow','difficult','complicated','confusing','painful','dreadful',
    'lousy','inferior','mediocre','failure','failed','problem','issue','wrong',
    'defective','unreliable','dangerous','harmful','toxic','rude','harsh',
    'angry','sad','miserable','upset','bitter','cheap','fragile','glitchy','ads'
}
INTENSIFIERS = {'very','extremely','absolutely','incredibly','really','so','much',
                'highly','totally','deeply','super','truly','quite','pretty'}
NEGATORS     = {'not','never','no','dont',"don't","doesn't",'doesnt','hardly','barely'}

def extract_features(text):
    if not text or not isinstance(text, str): return {'pos_score':5.0,'neg_score':0.0,'punct_score':0.0,'word_count':0}
    text_clean = re.sub(r'[^a-zA-Z\s\'!?]',' ', text.lower())
    words = text_clean.split()
    total = max(len(words), 1)
    pos_count = neg_count = 0.0
    negation_active = False; negation_window = 0
    for i, w in enumerate(words):
        if w in NEGATORS:    negation_active=True; negation_window=3
        else:
            negation_window -= 1
            if negation_window <= 0: negation_active=False
        boost = 1.6 if (i>0 and words[i-1] in INTENSIFIERS) else 1.0
        if w in POSITIVE_WORDS:
            neg_count+=boost if negation_active else 0; pos_count+=0 if negation_active else boost
        elif w in NEGATIVE_WORDS:
            pos_count+=boost if negation_active else 0; neg_count+=0 if negation_active else boost
    excl  = min(text.count('!')/max(total,1)*10, 1.0)
    caps  = sum(1 for c in text if c.isupper())/max(len(text),1)
    return {
        'pos_score':   min((pos_count/total)*10*3.0, 10.0),
        'neg_score':   min((neg_count/total)*10*3.0, 10.0),
        'punct_score': min((excl+caps)*5, 10.0),
        'word_count':  total
    }

print('✅ Feature extractor ready!')
# Quick demo
for t in ['Absolutely amazing tablet, love it!','Broken and slow, terrible product']:
    f = extract_features(t)
    print(f'  "{t[:50]}"')
    print(f'    pos={f["pos_score"]:.2f}  neg={f["neg_score"]:.2f}  punct={f["punct_score"]:.2f}')

## ✅ Step 8: Full Fuzzy Inference Pipeline

In [ ]:
def analyze_sentiment(text):
    features = extract_features(str(text))
    ps = np.clip(features['pos_score'],   0.01, 9.99)
    ns = np.clip(features['neg_score'],   0.01, 9.99)
    pu = np.clip(features['punct_score'], 0.01, 9.99)
    fis_sim.input['positive_score']    = float(ps)
    fis_sim.input['negative_score']    = float(ns)
    fis_sim.input['punctuation_score'] = float(pu)
    try:
        fis_sim.compute()
        score = float(fis_sim.output['sentiment'])
    except:
        score = float(np.clip((ps-ns+5)/10*50+25, 0, 100))
    if   score>=80: label,emoji='Very Positive','😄'
    elif score>=60: label,emoji='Positive','🙂'
    elif score>=40: label,emoji='Neutral','😐'
    elif score>=20: label,emoji='Negative','😟'
    else:           label,emoji='Very Negative','😠'
    su = sentiment_universe
    memberships = {
        'Very Negative': float(fuzz.interp_membership(su,sentiment['very_negative'].mf,score)),
        'Negative':      float(fuzz.interp_membership(su,sentiment['negative'].mf,score)),
        'Neutral':       float(fuzz.interp_membership(su,sentiment['neutral'].mf,score)),
        'Positive':      float(fuzz.interp_membership(su,sentiment['positive'].mf,score)),
        'Very Positive': float(fuzz.interp_membership(su,sentiment['very_positive'].mf,score)),
    }
    return {'text':text,'score':round(score,2),'label':label,'emoji':emoji,
            'features':features,'memberships':memberships}

print('✅ Inference pipeline ready!')

## ✅ Step 9: Apply Fuzzy FIS to Entire Dataset (Sample)

In [ ]:
from IPython.display import display
# Sample 2000 rows for analysis (full dataset takes a while — increase if desired)
SAMPLE_SIZE = 2000
df_sample = df.sample(SAMPLE_SIZE, random_state=42).copy()

print(f'🔄 Applying Fuzzy FIS to {SAMPLE_SIZE} reviews...')
results = []
for i, row in enumerate(df_sample.itertuples(), 1):
    r = analyze_sentiment(row.review_text)
    results.append({
        'review_text':  row.review_text[:80],
        'rating':       row.rating,
        'fuzzy_score':  r['score'],
        'fuzzy_label':  r['label'],
        'pos_input':    r['features']['pos_score'],
        'neg_input':    r['features']['neg_score'],
        'punct_input':  r['features']['punct_score'],
    })
    if i % 500 == 0: print(f'  ...{i}/{SAMPLE_SIZE} done')

df_results = pd.DataFrame(results)
print(f'\n✅ Done! Results shape: {df_results.shape}')
display(df_results.head(10))

## ✅ Step 10: Visualize Fuzzy Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fuzzy Sentiment Analysis — Dataset Results', fontsize=14, fontweight='bold', color='#38bdf8')

CMAP = {'Very Positive':'#22c55e','Positive':'#86efac','Neutral':'#facc15',
        'Negative':'#f97316','Very Negative':'#ef4444'}

# Fuzzy score histogram
ax = axes[0]
n,bins,patches = ax.hist(df_results['fuzzy_score'], bins=20, edgecolor='#0f172a', linewidth=1)
for patch,left in zip(patches,bins[:-1]):
    c = bins[1]-bins[0]; center=left+c/2
    if center>=80:  patch.set_facecolor('#22c55e')
    elif center>=60:patch.set_facecolor('#86efac')
    elif center>=40:patch.set_facecolor('#facc15')
    elif center>=20:patch.set_facecolor('#f97316')
    else:           patch.set_facecolor('#ef4444')
ax.set_title('Fuzzy Score Distribution',color='#38bdf8',fontweight='bold')
ax.set_xlabel('Fuzzy Score (0-100)'); ax.set_ylabel('Count'); ax.grid(True,alpha=0.3)

# Fuzzy label pie
ax = axes[1]
counts = df_results['fuzzy_label'].value_counts()
order  = ['Very Positive','Positive','Neutral','Negative','Very Negative']
counts = counts.reindex([o for o in order if o in counts.index])
ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
       colors=[CMAP[k] for k in counts.index],
       textprops={'color':'#e2e8f0'},wedgeprops={'edgecolor':'#0f172a','linewidth':2})
ax.set_title('Fuzzy Label Distribution',color='#38bdf8',fontweight='bold')

# Fuzzy score vs star rating (box plot)
ax = axes[2]
for rating, grp in df_results.groupby('rating'):
    if len(grp) > 10:
        bp = ax.boxplot(grp['fuzzy_score'], positions=[rating], widths=0.6,
                        patch_artist=True, notch=False,
                        boxprops=dict(facecolor='#38bdf8',alpha=0.5),
                        medianprops=dict(color='#facc15',linewidth=2),
                        whiskerprops=dict(color='#94a3b8'),
                        capprops=dict(color='#94a3b8'),
                        flierprops=dict(marker='o',color='#64748b',markersize=2,alpha=0.4))
ax.set_title('Fuzzy Score vs Star Rating',color='#38bdf8',fontweight='bold')
ax.set_xlabel('Star Rating'); ax.set_ylabel('Fuzzy Score')
ax.set_xticks([1,2,3,4,5]); ax.grid(True,alpha=0.3,axis='y')

plt.tight_layout()
plt.savefig('../outputs/dataset_results.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('✅ Saved: outputs/dataset_results.png')

## ✅ Step 11: Accuracy Evaluation (Fuzzy vs Star Rating)

In [ ]:
def rating_to_3class(r):
    if r>=4: return 'Positive'
    elif r==3: return 'Neutral'
    else: return 'Negative'

def fuzzy_to_3class(s):
    if s>=55: return 'Positive'
    elif s>=38: return 'Neutral'
    else: return 'Negative'

df_results['true_class']  = df_results['rating'].apply(rating_to_3class)
df_results['fuzzy_class'] = df_results['fuzzy_score'].apply(fuzzy_to_3class)

acc = accuracy_score(df_results['true_class'], df_results['fuzzy_class'])
print(f'✅ Fuzzy FIS Accuracy (vs Star Rating): {acc*100:.2f}%')
print()
print(classification_report(df_results['true_class'], df_results['fuzzy_class'],
                             target_names=['Negative','Neutral','Positive']))

In [ ]:
# Confusion Matrix
classes = ['Negative','Neutral','Positive']
cm = confusion_matrix(df_results['true_class'], df_results['fuzzy_class'], labels=classes)

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0f172a')
ax.set_facecolor('#1e293b')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes,
            ax=ax, linewidths=0.5, linecolor='#334155',
            annot_kws={'size':12,'fontweight':'bold'})
ax.set_title(f'Confusion Matrix — Accuracy: {acc*100:.1f}%',
             color='#38bdf8', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Predicted', color='#94a3b8')
ax.set_ylabel('Actual (Star Rating)', color='#94a3b8')
ax.tick_params(colors='#e2e8f0')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('✅ Saved: outputs/confusion_matrix.png')

## ✅ Step 12: FIS Control Surface (3D)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
N=18; xr=np.linspace(0.1,9.9,N); yr=np.linspace(0.1,9.9,N)
Z=np.zeros((N,N))
for i,xv in enumerate(xr):
    for j,yv in enumerate(yr):
        try:
            fis_sim.input['positive_score']=float(xv)
            fis_sim.input['negative_score']=float(yv)
            fis_sim.input['punctuation_score']=3.0
            fis_sim.compute(); Z[i,j]=fis_sim.output['sentiment']
        except: Z[i,j]=50.0
X,Y=np.meshgrid(xr,yr)
fig=plt.figure(figsize=(13,7),facecolor='#0f172a')
ax=fig.add_subplot(111,projection='3d'); ax.set_facecolor('#1e293b')
surf=ax.plot_surface(X,Y,Z.T,cmap='RdYlGn',alpha=0.88,edgecolor='none')
ax.set_xlabel('Positive Score',color='#94a3b8',labelpad=10)
ax.set_ylabel('Negative Score',color='#94a3b8',labelpad=10)
ax.set_zlabel('Sentiment Score',color='#94a3b8',labelpad=10)
ax.set_title('Mamdani FIS — 3D Control Surface',color='#38bdf8',fontweight='bold',fontsize=13,pad=20)
ax.tick_params(colors='#94a3b8')
plt.colorbar(surf,ax=ax,shrink=0.5,aspect=10,label='Sentiment Score')
plt.tight_layout()
plt.savefig('../outputs/surface_3d.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()
print('✅ Saved: outputs/surface_3d.png')

## ✅ Step 13: Top Products by Fuzzy Sentiment Score

In [ ]:
# Merge fuzzy results back to original sample
df_sample = df_sample.reset_index(drop=True)
df_sample['fuzzy_score'] = df_results['fuzzy_score'].values
df_sample['fuzzy_label'] = df_results['fuzzy_label'].values

# Aggregate by product
prod_agg = df_sample.groupby('product').agg(
    avg_fuzzy_score=('fuzzy_score','mean'),
    avg_rating=('rating','mean'),
    review_count=('fuzzy_score','count')
).reset_index().sort_values('avg_fuzzy_score', ascending=False)
prod_agg['product_short'] = prod_agg['product'].str[:45] + '...'
top10 = prod_agg.head(10)

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0f172a')
ax.set_facecolor('#1e293b')
bar_colors = ['#22c55e' if s>=80 else '#86efac' if s>=60 else '#facc15' 
              for s in top10['avg_fuzzy_score']]
bars = ax.barh(range(len(top10)), top10['avg_fuzzy_score'], color=bar_colors, edgecolor='#334155', height=0.65)
for i,(bar,score) in enumerate(zip(bars, top10['avg_fuzzy_score'])):
    ax.text(score+0.3, i, f'{score:.1f}', va='center', color='#e2e8f0', fontsize=9, fontweight='bold')
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10['product_short'], fontsize=8)
ax.set_xlabel('Average Fuzzy Sentiment Score', color='#94a3b8')
ax.set_title('Top 10 Products by Fuzzy Sentiment Score', color='#38bdf8', fontweight='bold', fontsize=13)
ax.set_xlim(0, 110); ax.grid(True, axis='x', alpha=0.3)
for sp in ax.spines.values(): sp.set_edgecolor('#334155')
plt.tight_layout()
plt.savefig('../outputs/top_products.png',dpi=150,bbox_inches='tight',facecolor='#0f172a')
plt.show()

display(prod_agg[['product_short','avg_fuzzy_score','avg_rating','review_count']].head(10).reset_index(drop=True))

## ✅ Step 14: Interactive Single Review Analysis

In [ ]:
# ✏️ CHANGE THIS TEXT AND RE-RUN THIS CELL
my_review = "The tablet is absolutely fantastic! Super fast, great screen, easy to use. My kids love it!"

r = analyze_sentiment(my_review)
print('┌─────────────────────────────────────────────────────────────┐')
print(f'│  {r["emoji"]}  Fuzzy Sentiment Analysis Result                       │')
print('├─────────────────────────────────────────────────────────────┤')
print(f'│  Review   : {str(r["text"])[:55]:55s}│')
print(f'│  Score    : {r["score"]:5.2f} / 100                                 │')
print(f'│  Sentiment: {r["label"]:49s}│')
print('├─────────────────────────────────────────────────────────────┤')
print(f'│  Fuzzy Inputs → pos:{r["features"]["pos_score"]:4.2f}  neg:{r["features"]["neg_score"]:4.2f}  punct:{r["features"]["punct_score"]:4.2f}          │')
print('├─────────────────────────────────────────────────────────────┤')
print('│  Membership Degrees:                                        │')
for lbl,deg in r['memberships'].items():
    bar='█'*int(deg*20); print(f'│   {lbl:14s}: {deg:.3f}  {bar:20s}             │')
print('└─────────────────────────────────────────────────────────────┘')

## ✅ Step 15: Save Final Results CSV

In [ ]:
df_results.to_csv('../outputs/fuzzy_results.csv', index=False)
print(f'✅ Results saved to outputs/fuzzy_results.csv')
print(f'   Rows: {len(df_results)}')
print(f'   Columns: {df_results.columns.tolist()}')
print()
print('📁 All output files:')
import os
for f in sorted(os.listdir('../outputs')):
    size = os.path.getsize(f'../outputs/{f}')
    print(f'   {f:<35} {size/1024:.1f} KB')

## ✅ Project Summary

---

| Item | Detail |
|---|---|
| **Dataset** | Amazon Product Reviews — 34,660 rows |
| **Analyzed** | 2,000 sampled reviews |
| **FIS Type** | Mamdani |
| **Rules** | 15 IF-THEN fuzzy rules |
| **MFs** | Triangular (trimf) |
| **Defuzz** | Centroid |
| **Output** | Score 0-100 → 5 sentiment levels |

### Pipeline:
```
Review Text → Feature Extraction → Fuzzification → Rule Evaluation → Aggregation → Defuzzification → Sentiment
```